# Retail Demand Forecasting — Day 2: Baseline Models
## ARIMA / SARIMA Benchmarks

**Objective:** Establish a statistical baseline using ARIMA and SARIMA per department.
Every model on Day 3–4 must beat these numbers.

**Resuming from:** `outputs/walmart_processed.csv` (Day 1 output)

**Models this notebook:**
- ARIMA(p,d,q) — univariate, no seasonality
- SARIMA(p,d,q)(P,D,Q,52) — captures annual cycle

**Holdout strategy:** Last 30 weeks per series (≈ 7.5 months)

**Evaluation metrics:** RMSE, MAE, MAPE

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
import os

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error

os.makedirs('outputs', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Holdout window ────────────────────────────────────────────
HOLDOUT = 30   # last 30 weeks reserved for evaluation

print('Libraries loaded.')
print(f'Holdout window: {HOLDOUT} weeks')

---
## 1. Load Data from Day 1

In [ ]:
df = pd.read_csv('outputs/walmart_processed.csv', parse_dates=['date'])

# Drop CPI and unemployment — excluded based on Day 1 CCF analysis
# cpi          → r=-0.081 at lag 11 — noise
# unemployment → r=+0.092 at lag 12 — Walmart counter-cyclical
# type, size   → store metadata, not used in time series models
df = df.drop(columns=['cpi', 'unemployment', 'type', 'size'])

STORES = df['store'].unique().tolist()
DEPTS  = df['dept'].unique().tolist()

print('Columns:', df.columns.tolist())
print('Shape  :', df.shape)
print('Stores :', STORES)
print('Depts  :', DEPTS)
print('Date range:', df['date'].min().date(), '→', df['date'].max().date())

---
## 2. Helper Functions

In [ ]:
# ── Metric calculations ──────────────────────────────────────
def compute_metrics(actual, predicted, model_name=''):
    """Returns RMSE, MAE, MAPE for a forecast vs actuals."""
    actual    = np.array(actual)
    predicted = np.array(predicted)

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    # MAPE — guard against zero actuals
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

    if model_name:
        print(f'  {model_name:20s} | RMSE={rmse:>10.2f} | MAE={mae:>10.2f} | MAPE={mape:>6.2f}%')
    return {'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'MAPE': round(mape, 2)}


# ── Series builder ────────────────────────────────────────────
def build_series(dept):
    """Aggregate weekly sales for one dept across all selected stores."""
    series = (
        df[df['dept'] == dept]
        .groupby('date')['weekly_sales']
        .sum()
        .asfreq('W-FRI')
        .ffill()
    )
    return series


# ── Train/test split ──────────────────────────────────────────
def split_series(series, holdout=HOLDOUT):
    return series.iloc[:-holdout], series.iloc[-holdout:]


print('Helper functions defined.')

---
## 3. ACF / PACF Analysis — Order Selection Guidance

> ACF and PACF plots guide the choice of p, d, q in ARIMA.
> 
> - **ACF tails off slowly** → AR process → look at PACF for p
> - **PACF cuts off at lag k** → AR(k) → set p=k
> - **ACF cuts off at lag k** → MA(k) → set q=k
> - **Both tail off** → ARMA — use AIC/BIC grid search

In [ ]:
fig, axes = plt.subplots(len(DEPTS), 2, figsize=(14, 3.5*len(DEPTS)))

for i, dept in enumerate(DEPTS):
    series = build_series(dept)
    train, _ = split_series(series)

    plot_acf(train,  lags=52, ax=axes[i][0], title=f'Dept {dept} — ACF',  alpha=0.05)
    plot_pacf(train, lags=52, ax=axes[i][1], title=f'Dept {dept} — PACF', alpha=0.05)

    axes[i][0].set_xlabel('Lag (weeks)')
    axes[i][1].set_xlabel('Lag (weeks)')

plt.suptitle('ACF / PACF — Training Series per Department', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/08_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/08_acf_pacf.png')
print()
print('Reading guide:')
print('  Shaded band = 95% confidence interval')
print('  Spikes outside band = statistically significant autocorrelation at that lag')
print('  Spikes at lags 52, 104 = annual seasonality confirmed')

---
## 4. ARIMA Baseline — Univariate, No Seasonality

> Uses `auto_arima` to search for best (p,d,q) order via AIC.
> This is the **weakest** baseline — no seasonal terms, no exogenous features.
> Every subsequent model must beat this.

In [ ]:
arima_results = {}

print('Fitting ARIMA per department...')
print('-' * 65)

for dept in DEPTS:
    series      = build_series(dept)
    train, test = split_series(series)

    # auto_arima — AIC-based order selection
    # seasonal=False here — pure ARIMA, seasonality handled in next section
    model = auto_arima(
        train,
        seasonal=False,
        information_criterion='aic',
        stepwise=True,
        suppress_warnings=True,
        error_action='ignore',
        max_p=5, max_q=5, max_d=2
    )

    preds = model.predict(n_periods=HOLDOUT)
    preds = np.maximum(preds, 0)   # clip negative forecasts — sales can't be negative

    metrics = compute_metrics(test, preds, model_name=f'Dept {dept}')
    arima_results[dept] = {
        'order'       : model.order,
        'AIC'         : round(model.aic(), 2),
        **metrics
    }

print()
print('ARIMA Orders selected by AIC:')
for dept, res in arima_results.items():
    print(f'  Dept {dept}: ARIMA{res["order"]} | AIC={res["AIC"]}')

---
## 5. SARIMA — Seasonal ARIMA with Annual Cycle

> Adds seasonal terms (P,D,Q) with period=52 weeks.
> Day 1 decomposition showed seasonal strength 0.818–0.971 across all depts.
> SARIMA should meaningfully outperform plain ARIMA.
>
> **Seasonal order search strategy:**
> - D=1 (seasonal differencing) for all depts — removes annual trend
> - P,Q searched over [0,1] — keep model parsimonious
> - p,d,q fixed from ARIMA step as starting point

In [ ]:
sarima_results = {}

print('Fitting SARIMA per department...')
print('(This takes 2-4 minutes — searching seasonal order space)')
print('-' * 65)

for dept in DEPTS:
    series      = build_series(dept)
    train, test = split_series(series)

    # auto_arima with seasonal=True, period=52
    model = auto_arima(
        train,
        seasonal=True,
        m=52,                        # annual period
        information_criterion='aic',
        stepwise=True,
        suppress_warnings=True,
        error_action='ignore',
        max_p=3, max_q=3, max_d=2,
        max_P=2, max_Q=2, max_D=1,
        D=1                          # force seasonal differencing — justified by decomposition
    )

    preds = model.predict(n_periods=HOLDOUT)
    preds = np.maximum(preds, 0)

    metrics = compute_metrics(test, preds, model_name=f'Dept {dept}')
    sarima_results[dept] = {
        'order'          : model.order,
        'seasonal_order' : model.seasonal_order,
        'AIC'            : round(model.aic(), 2),
        **metrics
    }

print()
print('SARIMA Orders selected by AIC:')
for dept, res in sarima_results.items():
    print(f'  Dept {dept}: ARIMA{res["order"]}x{res["seasonal_order"]} | AIC={res["AIC"]}')

---
## 6. ARIMA vs SARIMA — Comparison Table

In [ ]:
rows = []
for dept in DEPTS:
    a = arima_results[dept]
    s = sarima_results[dept]

    rmse_improvement = (a['RMSE'] - s['RMSE']) / a['RMSE'] * 100

    rows.append({
        'Dept'              : dept,
        'ARIMA Order'       : str(a['order']),
        'ARIMA RMSE'        : a['RMSE'],
        'SARIMA Order'      : f"{s['order']}x{s['seasonal_order']}",
        'SARIMA RMSE'       : s['RMSE'],
        'RMSE Improvement %': round(rmse_improvement, 1),
        'Winner'            : 'SARIMA' if s['RMSE'] < a['RMSE'] else 'ARIMA'
    })

comparison_df = pd.DataFrame(rows)
print('=' * 80)
print('ARIMA vs SARIMA — Holdout RMSE Comparison')
print('=' * 80)
print(comparison_df.to_string(index=False))
print()
print(f'SARIMA wins on {(comparison_df["Winner"]=="SARIMA").sum()}/{len(DEPTS)} departments')
print()
avg_improvement = comparison_df['RMSE Improvement %'].mean()
print(f'Average RMSE improvement SARIMA over ARIMA: {avg_improvement:.1f}%')
print()
print('Note: positive improvement % = SARIMA is better')
print('      negative improvement % = ARIMA is better (rare — check that dept)')

comparison_df.to_csv('outputs/day2_baseline_results.csv', index=False)
print('\nSaved → outputs/day2_baseline_results.csv')

---
## 7. Forecast Plots — Actuals vs ARIMA vs SARIMA

In [ ]:
fig, axes = plt.subplots(len(DEPTS), 1, figsize=(14, 5*len(DEPTS)))

for i, dept in enumerate(DEPTS):
    series      = build_series(dept)
    train, test = split_series(series)

    # Refit ARIMA
    arima_model = auto_arima(
        train, seasonal=False, stepwise=True,
        suppress_warnings=True, error_action='ignore',
        max_p=5, max_q=5
    )
    arima_preds = np.maximum(arima_model.predict(n_periods=HOLDOUT), 0)

    # Refit SARIMA
    sarima_model = auto_arima(
        train, seasonal=True, m=52, D=1, stepwise=True,
        suppress_warnings=True, error_action='ignore',
        max_p=3, max_q=3, max_P=2, max_Q=2
    )
    sarima_preds = np.maximum(sarima_model.predict(n_periods=HOLDOUT), 0)

    # Context window: last 52 weeks of train + full test
    context = train.iloc[-52:]
    test_index = test.index

    axes[i].plot(context.index, context.values,
                 color='steelblue', linewidth=1.0, label='Train (last 52w)')
    axes[i].plot(test_index, test.values,
                 color='black', linewidth=1.5, label='Actual')
    axes[i].plot(test_index, arima_preds,
                 color='darkorange', linewidth=1.2, linestyle='--', label=f'ARIMA  RMSE={arima_results[dept]["RMSE"]:,.0f}')
    axes[i].plot(test_index, sarima_preds,
                 color='seagreen', linewidth=1.2, linestyle='--', label=f'SARIMA RMSE={sarima_results[dept]["RMSE"]:,.0f}')

    axes[i].axvline(test_index[0], color='gray', linestyle=':', alpha=0.7, label='Holdout start')
    axes[i].set_title(f'Dept {dept} — Forecast vs Actuals', fontsize=12)
    axes[i].set_ylabel('Weekly Sales ($)')
    axes[i].legend(fontsize=8, loc='upper left')

plt.suptitle('ARIMA vs SARIMA Forecasts — 30-Week Holdout', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/09_forecast_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/09_forecast_comparison.png')

---
## 8. Residual Diagnostics — Best Model per Dept

> A well-specified model should have **white noise residuals**:
> - No remaining autocorrelation (Ljung-Box p > 0.05)
> - Roughly normally distributed
> - No systematic patterns
>
> If residuals fail — the model missed something. ARIMAX on Day 3 may fix it.

In [ ]:
print('Ljung-Box Test on SARIMA Residuals')
print('H0: Residuals are white noise | p > 0.05 → PASS (good fit)')
print('-' * 65)

residual_results = {}

fig, axes = plt.subplots(len(DEPTS), 2, figsize=(14, 3.5*len(DEPTS)))

for i, dept in enumerate(DEPTS):
    series      = build_series(dept)
    train, test = split_series(series)

    # Fit SARIMA using orders from earlier
    s   = sarima_results[dept]
    mod = SARIMAX(
        train,
        order=s['order'],
        seasonal_order=s['seasonal_order'],
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    fit = mod.fit(disp=False)
    residuals = fit.resid.dropna()

    # Ljung-Box test at lag 20
    lb = acorr_ljungbox(residuals, lags=[20], return_df=True)
    lb_p = lb['lb_pvalue'].iloc[0]
    verdict = '✅ PASS' if lb_p > 0.05 else '❌ FAIL — autocorrelation remains'

    residual_results[dept] = {'lb_p_lag20': round(lb_p, 4), 'verdict': verdict}
    print(f'  Dept {dept}: Ljung-Box p(lag=20) = {lb_p:.4f}  {verdict}')

    # Plot residuals + histogram
    axes[i][0].plot(residuals.index, residuals.values,
                    color='crimson', linewidth=0.7)
    axes[i][0].axhline(0, color='black', linewidth=0.8)
    axes[i][0].set_title(f'Dept {dept} — SARIMA Residuals', fontsize=9)
    axes[i][0].set_ylabel('Residual')

    axes[i][1].hist(residuals, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
    axes[i][1].set_title(f'Dept {dept} — Residual Distribution
Ljung-Box p={lb_p:.3f} {verdict}', fontsize=9)
    axes[i][1].set_xlabel('Residual value')

plt.suptitle('SARIMA Residual Diagnostics', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/10_residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved → outputs/10_residual_diagnostics.png')

---
## 9. Save Baseline — Reference for Day 3 Onwards

> Every model on Day 3 (ARIMAX), Day 4 (Prophet, LightGBM) is compared
> against this SARIMA baseline. Store it as a clean dict and CSV.

In [ ]:
# ── Build clean baseline reference ──────────────────────────
baseline = {}

for dept in DEPTS:
    a = arima_results[dept]
    s = sarima_results[dept]
    baseline[dept] = {
        'arima_rmse'  : a['RMSE'],
        'arima_mae'   : a['MAE'],
        'arima_mape'  : a['MAPE'],
        'sarima_rmse' : s['RMSE'],
        'sarima_mae'  : s['MAE'],
        'sarima_mape' : s['MAPE'],
        'sarima_order': str(s['order']),
        'sarima_seasonal_order': str(s['seasonal_order']),
        'lb_p_lag20'  : residual_results[dept]['lb_p_lag20'],
        'lb_verdict'  : residual_results[dept]['verdict']
    }

baseline_df = pd.DataFrame(baseline).T
baseline_df.index.name = 'dept'
baseline_df.to_csv('outputs/day2_sarima_baseline.csv')

print('=' * 65)
print('DAY 2 COMPLETE — Baseline Summary')
print('=' * 65)
print()
print('─── SARIMA Performance (Beat This on Day 3) ───')
print(baseline_df[['sarima_rmse','sarima_mae','sarima_mape','lb_verdict']].to_string())
print()
print('─── Day 3 ARIMAX Regressor Reminder ───')
print('  temperature    → ALL depts    (lag 0,  r=-0.439 from Day 1 CCF)')
print('  fuel_price     → ALL depts    (lag 1,  r=-0.163 from Day 1 CCF)')
print('  isholiday      → Dept 72 only (lag 0,  +94.7% holiday lift)')
print('  total_markdown → Depts 38, 92 (lag 0,  +10.2%, +7.2% lift)')
print()
print('─── Outputs ───')
for o in ['08_acf_pacf.png', '09_forecast_comparison.png',
          '10_residual_diagnostics.png', 'day2_baseline_results.csv',
          'day2_sarima_baseline.csv']:
    print(f'  outputs/{o}')